In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df[df['trafficSource.adwordsClickInfo.isVideoAd'].isnull() & df['trafficSource.adwordsClickInfo.page'].isnull() \
& df['trafficSource.adwordsClickInfo.adNetworkType'].isnull() & df['trafficSource.adwordsClickInfo.slot'].isnull()]\
[['trafficSource.adwordsClickInfo.isVideoAd','trafficSource.adwordsClickInfo.page','trafficSource.adwordsClickInfo.adNetworkType','trafficSource.adwordsClickInfo.slot']]

In [ ]:
df.drop(columns=['trafficSource.adContent','trafficSource.adwordsClickInfo.isVideoAd','trafficSource.adwordsClickInfo.adNetworkType',\
        'trafficSource.adwordsClickInfo.slot','trafficSource.adwordsClickInfo.page'],inplace=True)

In [ ]:
df['pageViews'].corr(df['totalHits'])

**Filling null values of rest of the columns**

In [ ]:
df['trafficSource.isTrueDirect'] = df['trafficSource.isTrueDirect'].fillna(False).astype(bool)
df['totals.bounces'] = df['totals.bounces'].fillna(0.0)
df['pageViews'] = df['pageViews'].fillna(df['totalHits'])
df['trafficSource.referralPath'] = df['trafficSource.referralPath'].fillna('unknown')
df['trafficSource.keyword'] = df['trafficSource.keyword'].fillna('unknown')
df['new_visits'] = df['new_visits'].fillna(0.0)

**No null values after filling the null values**

In [ ]:
df.isnull().sum().sort_values(ascending=False)

**All the columns which have value 'not available in demo dataset'**

In [ ]:
df.isin(['not available in demo dataset']).sum().sort_values(ascending=False)

**Deleted the columns with only value as 'not available in demo dataset'**

In [ ]:
df.drop(columns=['device.mobileDeviceMarketingName','device.screenColors','device.language','device.flashVersion',\
        'device.operatingSystemVersion','device.browserVersion','device.mobileDeviceModel','geoNetwork.networkLocation',\
        'device.mobileInputSelector','device.mobileDeviceBranding','browserMajor','device.screenResolution',\
        'device.browserSize'],inplace=True)

In [ ]:
df.loc[df['geoNetwork.city']=='not available in demo dataset','geoNetwork.city'] = 'unknown'
df.loc[df['geoNetwork.metro']=='not available in demo dataset','geoNetwork.metro'] = 'unknown'
df.loc[df['geoNetwork.region']=='not available in demo dataset','geoNetwork.region'] = 'unknown'

In [ ]:
df.isin(['not available in demo dataset']).sum().sort_values(ascending=False)

**Deleted the columns with only 1 unique value in the coluumns**

In [ ]:
df.drop(columns=['screenSize','totals.visits','socialEngagementType','locationZone'],inplace=True)

In [ ]:
df['purchaseValue'].describe()

In [ ]:
plt.hist(df['purchaseValue'],bins=50)

In [ ]:
plt.hist(np.log1p(df['purchaseValue']),bins=50)

In [ ]:
plt.hist(np.log1p(df[df['purchaseValue'] > 0]['purchaseValue']),bins=50)

**Created the column for the classification problem**

In [ ]:
df['zero_purch'] = np.where(df['purchaseValue']>0,1,0)

In [ ]:
df.drop(columns=['userId','sessionId','sessionStart'],inplace=True)

In [ ]:
bins = [0, 50, 100, 150, 250,np.inf]
df['pageViews_cat'] = pd.cut(df['pageViews'], bins=bins, labels=['A','B','C','D','E'])
df['totalHits_cat'] = pd.cut(df['totalHits'],bins=bins, labels=['A','B','C','D','E'])

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()
df['day'] = df['date'].dt.day
df['year_month'] = df['year'].astype(str)+'_'+df['month'].astype(str)


In [ ]:
df.drop(columns=['date'],inplace=True)

In [ ]:
num_cols_only=['pageViews','totalHits']
cat_cols=['trafficSource.isTrueDirect','geoCluster','geoNetwork.networkDomain','gclIdPresent','os','trafficSource.medium','totals.bounces','deviceType',\
'userChannel','geoNetwork.continent','device.isMobile','new_visits']
target_cols=['browser','trafficSource.campaign','geoNetwork.region','trafficSource',\
            'geoNetwork.subContinent','locationCountry','geoNetwork.city','geoNetwork.metro','sessionNumber',\
             'year','month','day_of_week','day','year_month','trafficSource.keyword','trafficSource.referralPath']
ordinal_enc = ['pageViews_cat','totalHits_cat']

In [ ]:
df_reg = df.copy()

**Classification problem**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test = train_test_split(df,test_size=0.2, random_state=7)

In [ ]:
X_train.shape,X_test.shape

In [ ]:
for col in target_cols:
    target_enc = pd.crosstab(X_train[col],X_train['zero_purch'])
    target_enc['perc1'] = target_enc[1]/(target_enc[0] + target_enc[1])
    X_train[col] = X_train[col].map(target_enc['perc1'])
    X_test[col] = X_test[col].map(target_enc['perc1'])
    X_test[col] = X_test[col].fillna(target_enc['perc1'].mean())

In [ ]:
X_train_cls = X_train.drop(columns=['purchaseValue','zero_purch'])
y_train_cls = X_train['zero_purch']
X_test_cls = X_test.drop(columns=['purchaseValue','zero_purch'])
y_test_cls = X_test['zero_purch']

In [ ]:
num_cols = num_cols_only + target_cols

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

#Numerical columns
scaler = StandardScaler()
X_train_cls[num_cols] = scaler.fit_transform(X_train_cls[num_cols])
X_test_cls[num_cols] = scaler.transform(X_test_cls[num_cols])

#Categorical columns
ohe = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

train_ohe = ohe.fit_transform(X_train_cls[cat_cols])
test_ohe = ohe.transform(X_test_cls[cat_cols])

ohe_cols = ohe.get_feature_names_out(cat_cols)


train_ohe_df = pd.DataFrame(train_ohe, columns=ohe_cols, index=X_train_cls.index)
test_ohe_df = pd.DataFrame(test_ohe, columns=ohe_cols, index=X_test_cls.index)

X_train_cls = X_train_cls.drop(columns=cat_cols)
X_test_cls = X_test_cls.drop(columns=cat_cols)

#Ordinal Encoding
encoder = OrdinalEncoder(categories=[['A','B','C','D','E'],['A','B','C','D','E']])
train_ordinal=encoder.fit_transform(X_train_cls[['pageViews_cat','totalHits_cat']])
test_ordinal=encoder.transform(X_test_cls[['pageViews_cat','totalHits_cat']])

train_ordinal_enc = pd.DataFrame(train_ordinal,columns=ordinal_enc,index=X_train_cls.index)
test_ordinal_enc = pd.DataFrame(test_ordinal,columns=ordinal_enc,index=X_test_cls.index)

X_train_cls = X_train_cls.drop(columns=ordinal_enc)
X_test_cls = X_test_cls.drop(columns=ordinal_enc)

# Combining all the columns together
X_train_cls = pd.concat([X_train_cls, train_ohe_df,train_ordinal_enc], axis=1)
X_test_cls = pd.concat([X_test_cls, test_ohe_df,test_ordinal_enc], axis=1)

In [ ]:
X_test_cls = X_test_cls[X_train_cls.columns] # to get train and test columns in the same order.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


dt = DecisionTreeClassifier(class_weight='balanced', random_state=42)

param_grid = {
    'max_depth': [15,None],
    'min_samples_split': [2, 5],
    'criterion': ['gini', 'entropy']
}


grid = GridSearchCV(dt, param_grid, scoring='accuracy', cv=3, n_jobs=-1)
grid.fit(X_train_cls, y_train_cls)


best_dt = grid.best_estimator_
print("Best Parameters:", grid.best_params_)

# Predictions for both test and train data
y_train_pred = best_dt.predict(X_train_cls)
y_test_pred = best_dt.predict(X_test_cls)

# Accuracy for both test and train data
train_acc = accuracy_score(y_train_cls, y_train_pred)
test_acc = accuracy_score(y_test_cls, y_test_pred)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion Matrix Train data
conf_matrix_train = confusion_matrix(y_train_cls, y_train_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix_train, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Train')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Confusion Matrix Test Data
conf_matrix_test = confusion_matrix(y_test_cls, y_test_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix_test, annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - Test')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
importances = best_dt.feature_importances_

feature_names = X_train_cls.columns

# Create a DataFrame for better readability
feat_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Show top features
print(feat_importance_df.head(20))

In [ ]:
from sklearn.linear_model import LogisticRegression


log_reg = LogisticRegression(class_weight='balanced', max_iter=1000)

param_grid = {
    'C': [5, 10, 15],
    'penalty': ['l1', 'l2']
}

grid = GridSearchCV(log_reg, param_grid, scoring='accuracy', cv=3, n_jobs=-1)
grid.fit(X_train_cls, y_train_cls)


best_dt = grid.best_estimator_
print("Best Parameters:", grid.best_params_)

# Predictions
y_train_pred = best_dt.predict(X_train_cls)
y_test_pred = best_dt.predict(X_test_cls)

# Accuracy
train_acc = accuracy_score(y_train_cls, y_train_pred)
test_acc = accuracy_score(y_test_cls, y_test_pred)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion Matrix Trian
conf_matrix_train = confusion_matrix(y_train_cls, y_train_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix_train, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Train')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Confusion Matrix Test
conf_matrix_test = confusion_matrix(y_test_cls, y_test_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix_test, annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - Test')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

scale_pos_weight = np.sum(y_train_cls == 0) / np.sum(y_train_cls == 1)

# Define XGBoost model
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Hyperparameter grid
param_grid = {
    'n_estimators': [350,400],
    'max_depth': [11,12],
    'learning_rate': [0.1],
    'scale_pos_weight': [scale_pos_weight],  # important for imbalance
    'colsample_bytree':[0.6],
    'colsample_bylevel':[0.7],
    'subsample':[0.8,0.9]
}

# Perform GridSearchCV
grid = GridSearchCV(xgb, param_grid, scoring='accuracy', cv=3, n_jobs=-1, verbose=3)
grid.fit(X_train_cls, y_train_cls)

# Best estimator
best_xgb = grid.best_estimator_
print("Best Parameters:", grid.best_params_)

# Predictions
y_train_pred = best_xgb.predict(X_train_cls)
y_test_pred = best_xgb.predict(X_test_cls)

# Accuracy
train_acc = accuracy_score(y_train_cls, y_train_pred)
test_acc = accuracy_score(y_test_cls, y_test_pred)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion Matrix - Train
conf_matrix_train = confusion_matrix(y_train_cls, y_train_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix_train, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Train')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Confusion Matrix - Test
conf_matrix_test = confusion_matrix(y_test_cls, y_test_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix_test, annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - Test')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(random_state=42,bootstrap=True, max_samples=0.7, max_features='sqrt')

# Define parameter grid
param_grid = {
    'n_estimators': [200,250],
    'max_depth': [10, 20, 15],
    'min_samples_split': [2],
    'min_samples_leaf': [1]
}

# GridSearchCV
grid_search = GridSearchCV(estimator=rf,
                           param_grid=param_grid,
                           cv=3,
                           n_jobs=-1,
                           verbose=2,
                           scoring='accuracy')

# Fit
grid_search.fit(X_train_cls, y_train_cls)

# Best estimator
best_rfc = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

# Predictions
y_train_pred = best_rfc.predict(X_train_cls)
y_test_pred = best_rfc.predict(X_test_cls)

# Accuracy
train_acc = accuracy_score(y_train_cls, y_train_pred)
test_acc = accuracy_score(y_test_cls, y_test_pred)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion Matrix - Train
conf_matrix_train = confusion_matrix(y_train_cls, y_train_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix_train, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Train')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Confusion Matrix - Test
conf_matrix_test = confusion_matrix(y_test_cls, y_test_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix_test, annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - Test')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

**Regression Problem**

In [ ]:
df = pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv")



df['pageViews']=df['pageViews'].fillna(df['totalHits'])
df['new_visits'] = df['new_visits'].fillna(0.0)
df['totals.bounces'] = df['totals.bounces'].fillna(0.0)
df['trafficSource.keyword'] = df['trafficSource.keyword'].fillna('(not provided)')
df['trafficSource.isTrueDirect'] = df['trafficSource.isTrueDirect'].fillna('False')
df['trafficSource.referralPath'] = df['trafficSource.referralPath'].fillna('/')

df.drop(columns = ['trafficSource.adwordsClickInfo.page','trafficSource.adwordsClickInfo.slot', \
                   'trafficSource.adwordsClickInfo.adNetworkType','trafficSource.adwordsClickInfo.isVideoAd'],inplace=True)
df.drop(columns = ['trafficSource.adContent'],inplace=True)
df.drop(columns = ['geoNetwork.networkLocation','device.flashVersion','device.screenColors',\
                'device.screenResolution','device.browserVersion','device.mobileDeviceMarketingName','device.browserSize',\
        'device.mobileDeviceBranding','device.browserSize','device.mobileDeviceBranding','device.mobileInputSelector','device.mobileDeviceModel',\
        'browserMajor','device.operatingSystemVersion','device.language','screenSize'],inplace=True)
df.drop(columns=['sessionStart'],inplace=True)
df.drop(columns = ['totals.visits', 'socialEngagementType', 'locationZone'],inplace=True)

df['trafficSource.isTrueDirect']=df['trafficSource.isTrueDirect'].astype(str)

df['device.isMobile']=df['device.isMobile'].astype(str)

df = df[df['purchaseValue'] > 0]

train_df_reg,test_df_reg = train_test_split(df,random_state=42)

column = ['trafficSource.keyword','sessionId','userId','trafficSource.campaign','sessionNumber', \
       'geoNetwork.region','trafficSource','locationCountry', 'geoNetwork.city', 'geoNetwork.metro',\
       'trafficSource.referralPath','date','deviceType','userChannel']
for col in column:
    target_enc=train_df_reg.groupby([col])['purchaseValue'].mean()
    train_df_reg[col]=train_df_reg[col].map(target_enc)
    test_df_reg[col] = test_df_reg[col].map(target_enc)
    test_df_reg[col] = test_df_reg[col].fillna(target_enc.mean())

In [ ]:
X_train_reg = train_df_reg.drop(columns = ['purchaseValue'])
y_train_reg = train_df_reg['purchaseValue']
X_test_reg = test_df_reg.drop(columns = ['purchaseValue'])
y_test_reg = test_df_reg['purchaseValue']

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_cols = ['trafficSource.keyword','userId','trafficSource.campaign','gclIdPresent','sessionNumber',\
'geoNetwork.region','trafficSource','sessionId','locationCountry','geoNetwork.city','geoNetwork.metro','pageViews',\
'trafficSource.referralPath','totals.bounces','date','deviceType','userChannel','totalHits','new_visits']

categorical_cols = ['trafficSource.isTrueDirect','browser','geoCluster','geoNetwork.networkDomain','os','geoNetwork.subContinent',\
'trafficSource.medium','geoNetwork.continent','device.isMobile']

# --- NUMERIC PROCESSING ---
scaler = StandardScaler()
X_train_reg[numeric_cols] = scaler.fit_transform(X_train_reg[numeric_cols])
X_test_reg[numeric_cols] = scaler.transform(X_test_reg[numeric_cols])

# --- CATEGORICAL PROCESSING ---
ohe = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

# Fit and transform
train_ohe = ohe.fit_transform(X_train_reg[categorical_cols])
test_ohe = ohe.transform(X_test_reg[categorical_cols])

# Get new column names
ohe_cols = ohe.get_feature_names_out(categorical_cols)

# Create new DataFrames from the encoded arrays
train_ohe_df = pd.DataFrame(train_ohe, columns=ohe_cols, index=X_train_reg.index)
test_ohe_df = pd.DataFrame(test_ohe, columns=ohe_cols, index=X_test_reg.index)

# Drop original categorical columns from X
X_train_reg = X_train_reg.drop(columns=categorical_cols)
X_test_reg = X_test_reg.drop(columns=categorical_cols)

# Combine numeric and encoded categorical features
X_train_reg = pd.concat([X_train_reg, train_ohe_df], axis=1)
X_test_reg = pd.concat([X_test_reg, test_ohe_df], axis=1)


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error

dt = DecisionTreeRegressor(random_state=42)

param_grid = {
    'max_depth': [3, 5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}


grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, 
                           cv=5, scoring='r2', n_jobs=-1, verbose=1)
y_train_reg_log = np.log1p(y_train_reg)
y_test_reg_log = np.log1p(y_test_reg)


grid_search.fit(X_train_reg, y_train_reg_log)


best_dt = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

# Predict
y_train_pred = best_dt.predict(X_train_reg)
y_test_pred = best_dt.predict(X_test_reg)

y_train_pred = np.expm1(y_train_pred)
y_test_pred = np.expm1(y_test_pred)

# Evaluation metrics
train_r2 = r2_score(y_train_reg, y_train_pred)
test_r2 = r2_score(y_test_reg, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train_reg, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_test_pred))

print("\nTraining R² Score:", train_r2)
print("Test R² Score:", test_r2)

print("Training RMSE:", train_rmse)
print("Test RMSE:", test_rmse)


In [ ]:
from xgboost import XGBRegressor


xgb = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)


param_grid = {
    'n_estimators': [150,200,250],
    'max_depth': [2,3,4],
    'learning_rate': [0.1,0.05],
    'subsample': [0.3],
    'colsample_bytree': [0.6]
}


grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, 
                           cv=3, scoring='r2', n_jobs=-1, verbose=1)

y_train_reg_log = np.log1p(y_train_reg)
y_test_reg_log = np.log1p(y_test_reg)


grid_search.fit(X_train_reg, y_train_reg_log)


best_xgb = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

# Predict
y_train_pred = best_xgb.predict(X_train_reg)
y_test_pred = best_xgb.predict(X_test_reg)

y_train_pred = np.expm1(y_train_pred)
y_test_pred = np.expm1(y_test_pred)

# Evaluation metrics
train_r2 = r2_score(y_train_reg, y_train_pred)
test_r2 = r2_score(y_test_reg, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train_reg, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_test_pred))

print("\nTraining R² Score:", train_r2)
print("Test R² Score:", test_r2)

print("Training RMSE:", train_rmse)
print("Test RMSE:", test_rmse)


**Training and Testing data with full data**

In [ ]:
df = pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv")
df_actual=pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv")

In [ ]:
df.drop(columns=['trafficSource.adContent','trafficSource.adwordsClickInfo.isVideoAd','trafficSource.adwordsClickInfo.adNetworkType',\
        'trafficSource.adwordsClickInfo.slot','trafficSource.adwordsClickInfo.page'],inplace=True)
df_actual.drop(columns=['trafficSource.adContent','trafficSource.adwordsClickInfo.isVideoAd','trafficSource.adwordsClickInfo.adNetworkType',\
        'trafficSource.adwordsClickInfo.slot','trafficSource.adwordsClickInfo.page'],inplace=True)

In [ ]:
df['trafficSource.isTrueDirect'] = df['trafficSource.isTrueDirect'].fillna(False).astype(bool)
df['totals.bounces'] = df['totals.bounces'].fillna(0.0)
df['totals.bounces'] = df['totals.bounces'].fillna(0.0)
df['pageViews'] = df['pageViews'].fillna(df['totalHits'])
df['trafficSource.referralPath'] = df['trafficSource.referralPath'].fillna('unknown')
df['trafficSource.keyword'] = df['trafficSource.keyword'].fillna('unknown')
df['new_visits'] = df['new_visits'].fillna(0.0)
df.drop(columns=['device.mobileDeviceMarketingName','device.screenColors','device.language','device.flashVersion',\
        'device.operatingSystemVersion','device.browserVersion','device.mobileDeviceModel','geoNetwork.networkLocation',\
        'device.mobileInputSelector','device.mobileDeviceBranding','browserMajor','device.screenResolution',\
        'device.browserSize'],inplace=True)
df.loc[df['geoNetwork.city']=='not available in demo dataset','geoNetwork.city'] = 'unknown'
df.loc[df['geoNetwork.metro']=='not available in demo dataset','geoNetwork.metro'] = 'unknown'
df.loc[df['geoNetwork.region']=='not available in demo dataset','geoNetwork.region'] = 'unknown'

df_actual['trafficSource.isTrueDirect'] = df_actual['trafficSource.isTrueDirect'].fillna(False).astype(bool)
df_actual['totals.bounces'] = df_actual['totals.bounces'].fillna(0.0)
df_actual['totals.bounces'] = df_actual['totals.bounces'].fillna(0.0)
df_actual['pageViews'] = df_actual['pageViews'].fillna(df_actual['totalHits'])
df_actual['trafficSource.referralPath'] = df_actual['trafficSource.referralPath'].fillna('unknown')
df_actual['trafficSource.keyword'] = df_actual['trafficSource.keyword'].fillna('unknown')
df_actual['new_visits'] = df_actual['new_visits'].fillna(0.0)
df_actual.drop(columns=['device.mobileDeviceMarketingName','device.screenColors','device.language','device.flashVersion',\
        'device.operatingSystemVersion','device.browserVersion','device.mobileDeviceModel','geoNetwork.networkLocation',\
        'device.mobileInputSelector','device.mobileDeviceBranding','browserMajor','device.screenResolution',\
        'device.browserSize'],inplace=True)
df_actual.loc[df_actual['geoNetwork.city']=='not available in demo dataset','geoNetwork.city'] = 'unknown'
df_actual.loc[df_actual['geoNetwork.metro']=='not available in demo dataset','geoNetwork.metro'] = 'unknown'
df_actual.loc[df_actual['geoNetwork.region']=='not available in demo dataset','geoNetwork.region'] = 'unknown'

In [ ]:
df.drop(columns=['screenSize','totals.visits','socialEngagementType','locationZone'],inplace=True)
df_actual.drop(columns=['screenSize','totals.visits','socialEngagementType','locationZone'],inplace=True)

In [ ]:
df['zero_purch'] = np.where(df['purchaseValue']>0,1,0)

In [ ]:
df.drop(columns=['userId','sessionId','sessionStart'],inplace=True)
df_actual.drop(columns=['userId','sessionId','sessionStart'],inplace=True)

In [ ]:
df.drop_duplicates(inplace=True)
# df_actual.drop_duplicates(inplace=True)

In [ ]:
bins = [0, 50, 100, 150, 250,np.inf]
df['pageViews_cat'] = pd.cut(df['pageViews'], bins=bins, labels=['A','B','C','D','E'])
df['totalHits_cat'] = pd.cut(df['totalHits'],bins=bins, labels=['A','B','C','D','E'])
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()
df['day'] = df['date'].dt.day
df['year_month'] = df['year'].astype(str)+'_'+df['month'].astype(str)
df['pageViews'] = np.log1p(df['pageViews'])
df['totalHits'] = np.log1p(df['totalHits'])
df.drop(columns=['date'],inplace=True)

bins = [0, 50, 100, 150, 250,np.inf]
df_actual['pageViews_cat'] = pd.cut(df_actual['pageViews'], bins=bins, labels=['A','B','C','D','E'])
df_actual['totalHits_cat'] = pd.cut(df_actual['totalHits'],bins=bins, labels=['A','B','C','D','E'])
df_actual['date'] = pd.to_datetime(df_actual['date'], format='%Y%m%d')
df_actual['year'] = df_actual['date'].dt.year
df_actual['month'] = df_actual['date'].dt.month
df_actual['day_of_week'] = df_actual['date'].dt.day_name()
df_actual['day'] = df_actual['date'].dt.day
df_actual['year_month'] = df_actual['year'].astype(str)+'_'+df_actual['month'].astype(str)
df_actual['pageViews'] = np.log1p(df_actual['pageViews'])
df_actual['totalHits'] = np.log1p(df_actual['totalHits'])
df_actual.drop(columns=['date'],inplace=True)

In [ ]:
num_cols_only=['pageViews','totalHits']
cat_cols=['trafficSource.isTrueDirect','geoCluster','geoNetwork.networkDomain','gclIdPresent','os','trafficSource.medium','totals.bounces','deviceType',\
'userChannel','geoNetwork.continent','device.isMobile','new_visits']
target_cols=['browser','trafficSource.campaign','geoNetwork.region','trafficSource',\
            'geoNetwork.subContinent','locationCountry','geoNetwork.city','geoNetwork.metro','sessionNumber',\
             'year','month','day_of_week','day','year_month','trafficSource.keyword','trafficSource.referralPath']
ordinal_enc = ['pageViews_cat','totalHits_cat']

In [ ]:
df_reg = df.copy()
df_reg_actual=df_actual.copy()

In [ ]:
for col in target_cols:
    target_enc = pd.crosstab(df[col],df['zero_purch'])
    target_enc['perc1'] = target_enc[1]/(target_enc[0] + target_enc[1])
    df[col] = df[col].map(target_enc['perc1'])
    df_actual[col] = df_actual[col].map(target_enc['perc1'])
    df_actual[col] = df_actual[col].fillna(target_enc['perc1'].mean())

X_train_cls = df.drop(columns=['purchaseValue','zero_purch'])
y_train_cls = df['zero_purch']
X_test_cls = df_actual.copy()

num_cols = num_cols_only + target_cols


from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

#Numerical columns
scaler = StandardScaler()
X_train_cls[num_cols] = scaler.fit_transform(X_train_cls[num_cols])
X_test_cls[num_cols] = scaler.transform(X_test_cls[num_cols])

#Categorical columns
ohe = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

train_ohe = ohe.fit_transform(X_train_cls[cat_cols])
test_ohe = ohe.transform(X_test_cls[cat_cols])

ohe_cols = ohe.get_feature_names_out(cat_cols)


train_ohe_df = pd.DataFrame(train_ohe, columns=ohe_cols, index=X_train_cls.index)
test_ohe_df = pd.DataFrame(test_ohe, columns=ohe_cols, index=X_test_cls.index)

X_train_cls = X_train_cls.drop(columns=cat_cols)
X_test_cls = X_test_cls.drop(columns=cat_cols)

#Ordinal Encoding
encoder = OrdinalEncoder(categories=[['A','B','C','D','E'],['A','B','C','D','E']])
train_ordinal=encoder.fit_transform(X_train_cls[['pageViews_cat','totalHits_cat']])
test_ordinal=encoder.transform(X_test_cls[['pageViews_cat','totalHits_cat']])

train_ordinal_enc = pd.DataFrame(train_ordinal,columns=ordinal_enc,index=X_train_cls.index)
test_ordinal_enc = pd.DataFrame(test_ordinal,columns=ordinal_enc,index=X_test_cls.index)

X_train_cls = X_train_cls.drop(columns=ordinal_enc)
X_test_cls = X_test_cls.drop(columns=ordinal_enc)

# Combining all the columns together
X_train_cls = pd.concat([X_train_cls, train_ohe_df,train_ordinal_enc], axis=1)
X_test_cls = pd.concat([X_test_cls, test_ohe_df,test_ordinal_enc], axis=1)

In [ ]:
X_test_cls = X_test_cls[X_train_cls.columns]

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

scale_pos_weight = np.sum(y_train_cls == 0) / np.sum(y_train_cls == 1)


xgboost = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, colsample_bylevel = 0.7, colsample_bytree = 0.6, \
                        learning_rate = 0.1, max_depth=12, n_estimators =350, scale_pos_weight = scale_pos_weight,subsample=0.9)


xgboost.fit(X_train_cls, y_train_cls)


# Predictions
y_train_pred = xgboost.predict(X_train_cls)
y_test_pred = xgboost.predict(X_test_cls)

# Accuracy
train_acc = accuracy_score(y_train_cls, y_train_pred)
print(f"Train Accuracy: {train_acc:.4f}")


# Confusion Matrix - Train
conf_matrix_train = confusion_matrix(y_train_cls, y_train_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix_train, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Train')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
df = pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv")
df_reg_actual=pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv")

df_actual = df_reg_actual[y_test_pred==1]


df['pageViews']=df['pageViews'].fillna(df['totalHits'])
df['new_visits'] = df['new_visits'].fillna(0.0)
df['totals.bounces'] = df['totals.bounces'].fillna(0.0)
df['trafficSource.keyword'] = df['trafficSource.keyword'].fillna('(not provided)')
df['trafficSource.isTrueDirect'] = df['trafficSource.isTrueDirect'].fillna('False')
df['trafficSource.referralPath'] = df['trafficSource.referralPath'].fillna('/')
df.drop(columns = ['trafficSource.adwordsClickInfo.page','trafficSource.adwordsClickInfo.slot', \
                   'trafficSource.adwordsClickInfo.adNetworkType','trafficSource.adwordsClickInfo.isVideoAd'],inplace=True)
df.drop(columns = ['trafficSource.adContent'],inplace=True)
df.drop(columns = ['geoNetwork.networkLocation','device.flashVersion','device.screenColors',\
                'device.screenResolution','device.browserVersion','device.mobileDeviceMarketingName','device.browserSize',\
        'device.mobileDeviceBranding','device.browserSize','device.mobileDeviceBranding','device.mobileInputSelector','device.mobileDeviceModel',\
        'browserMajor','device.operatingSystemVersion','device.language','screenSize'],inplace=True)
df.drop(columns=['sessionStart'],inplace=True)
df.drop(columns = ['totals.visits', 'socialEngagementType', 'locationZone'],inplace=True)


df_actual['pageViews']=df_actual['pageViews'].fillna(df['totalHits'])
df_actual['new_visits'] = df_actual['new_visits'].fillna(0.0)
df_actual['totals.bounces'] = df_actual['totals.bounces'].fillna(0.0)
df_actual['trafficSource.keyword'] = df_actual['trafficSource.keyword'].fillna('(not provided)')
df_actual['trafficSource.isTrueDirect'] = df_actual['trafficSource.isTrueDirect'].fillna('False')
df_actual['trafficSource.referralPath'] = df_actual['trafficSource.referralPath'].fillna('/')
df_actual.drop(columns = ['trafficSource.adwordsClickInfo.page','trafficSource.adwordsClickInfo.slot', \
                   'trafficSource.adwordsClickInfo.adNetworkType','trafficSource.adwordsClickInfo.isVideoAd'],inplace=True)
df_actual.drop(columns = ['trafficSource.adContent'],inplace=True)
df_actual.drop(columns = ['geoNetwork.networkLocation','device.flashVersion','device.screenColors',\
                'device.screenResolution','device.browserVersion','device.mobileDeviceMarketingName','device.browserSize',\
        'device.mobileDeviceBranding','device.browserSize','device.mobileDeviceBranding','device.mobileInputSelector','device.mobileDeviceModel',\
        'browserMajor','device.operatingSystemVersion','device.language','screenSize'],inplace=True)
df_actual.drop(columns=['sessionStart'],inplace=True)
df_actual.drop(columns = ['totals.visits', 'socialEngagementType', 'locationZone'],inplace=True)



column = ['trafficSource.keyword','sessionId','userId','trafficSource.campaign','sessionNumber', \
       'geoNetwork.region','trafficSource','locationCountry', 'geoNetwork.city', 'geoNetwork.metro',\
       'trafficSource.referralPath','date','deviceType','userChannel']
for col in column:
    target_enc=df.groupby([col])['purchaseValue'].mean()
    df[col]=df[col].map(target_enc)
    df_actual[col] = df_actual[col].map(target_enc)
    df_actual[col] = df_actual[col].fillna(target_enc.mean())

df['trafficSource.isTrueDirect']=df['trafficSource.isTrueDirect'].astype(str)
df_actual['trafficSource.isTrueDirect']=df_actual['trafficSource.isTrueDirect'].astype(str)

df['device.isMobile']=df['device.isMobile'].astype(str)
df_actual['device.isMobile']=df_actual['device.isMobile'].astype(str)

X_train_reg = df.drop(columns = ['purchaseValue'])
y_train_reg = df['purchaseValue']
X_test_reg = df_actual.copy()



from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_cols = ['trafficSource.keyword','userId','trafficSource.campaign','gclIdPresent','sessionNumber',\
'geoNetwork.region','trafficSource','sessionId','locationCountry','geoNetwork.city','geoNetwork.metro','pageViews',\
'trafficSource.referralPath','totals.bounces','date','deviceType','userChannel','totalHits','new_visits']

categorical_cols = ['trafficSource.isTrueDirect','browser','geoCluster','geoNetwork.networkDomain','os','geoNetwork.subContinent',\
'trafficSource.medium','geoNetwork.continent','device.isMobile']

# --- NUMERIC PROCESSING ---
scaler = StandardScaler()
X_train_reg[numeric_cols] = scaler.fit_transform(X_train_reg[numeric_cols])
X_test_reg[numeric_cols] = scaler.transform(X_test_reg[numeric_cols])

# --- CATEGORICAL PROCESSING ---
ohe = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

# Fit and transform
train_ohe = ohe.fit_transform(X_train_reg[categorical_cols])
test_ohe = ohe.transform(X_test_reg[categorical_cols])

# Get new column names
ohe_cols = ohe.get_feature_names_out(categorical_cols)

# Create new DataFrames from the encoded arrays
train_ohe_df = pd.DataFrame(train_ohe, columns=ohe_cols, index=X_train_reg.index)
test_ohe_df = pd.DataFrame(test_ohe, columns=ohe_cols, index=X_test_reg.index)

# Drop original categorical columns from X
X_train_reg = X_train_reg.drop(columns=categorical_cols)
X_test_reg = X_test_reg.drop(columns=categorical_cols)

# Combine numeric and encoded categorical features
X_train_reg = pd.concat([X_train_reg, train_ohe_df], axis=1)
X_test_reg = pd.concat([X_test_reg, test_ohe_df], axis=1)

X_test_reg = X_test_reg[X_train_reg.columns]

from xgboost import XGBRegressor
from sklearn.metrics import r2_score

model = XGBRegressor(objective='reg:squarederror', random_state=42, colsample_bytree = 0.6, \
                    learning_rate= 0.1, max_depth=2, n_estimators = 200, subsample= 0.3)

model.fit(X_train_reg,y_train_reg)
y_pred_train_reg = model.predict(X_train_reg)
y_pred_test_reg = model.predict(X_test_reg)

print("R2_score training",r2_score(y_train_reg,y_pred_train_reg))



In [ ]:
final_p = X_test_reg.copy()
final_p['prediction'] = y_pred_test_reg
all_p = df_reg_actual.copy()
all_p['predict'] = 0
all_p.loc[final_p.index, 'predict'] = final_p['prediction']

In [ ]:
pd.DataFrame({'id':np.arange(df_reg_actual.shape[0]),'purchaseValue':all_p['predict']}).to_csv('submission.csv',index=False)